# Run and inspect Spark
Run beside the matching Markdown runbook. Start with a fresh Python kernel. Keep the Spark session alive until the inspection checkpoint. Configuration is read from config.json beside this notebook.


In [ ]:
import json, os, socket, uuid
from pathlib import Path
from pyspark.sql import SparkSession
config = json.loads(Path('config.json').read_text())
required = config['credential_environment']
missing = [name for name in required.values() if not os.environ.get(name)]
assert not missing, 'Missing configured credential variables: ' + ', '.join(missing)
credentials = {key: os.environ[name] for key, name in required.items()}
run_id = 'ops_' + uuid.uuid4().hex[:12]
root = f"s3a://{config['bucket']}/{config['temporary_prefix']}/{run_id}"
print('Run:', run_id, 'Temporary data:', root)
# Values remain in memory; never display credentials or the full Spark configuration.


In [ ]:
builder = (SparkSession.builder.appName(run_id)
    .master(config['spark_master'])
    .config('spark.driver.host', os.environ.get('POD_IP') or socket.gethostbyname(socket.gethostname()))
    .config('spark.driver.bindAddress', '0.0.0.0')
    .config('spark.ui.port', '4040')
    .config('spark.cores.max', '2')
    .config('spark.executor.cores', '1')
    .config('spark.executor.memory', '512m')
    .config('spark.sql.shuffle.partitions', '4')
    .config('spark.eventLog.enabled', 'true')
    .config('spark.eventLog.dir', config['event_log_directory'])
    .config('spark.redaction.regex', '(?i)secret|password|token|access.?key|credential')
    .config('spark.hadoop.fs.s3a.endpoint', config['s3_endpoint'])
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .config('spark.hadoop.fs.s3a.access.key', credentials['s3_access_key'])
    .config('spark.hadoop.fs.s3a.secret.key', credentials['s3_secret_key'])
    .config('spark.sql.catalog.polaris', 'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.polaris.type', 'rest')
    .config('spark.sql.catalog.polaris.uri', config['polaris_uri'])
    .config('spark.sql.catalog.polaris.warehouse', config['polaris_warehouse'])
    .config('spark.sql.catalog.polaris.scope', 'PRINCIPAL_ROLE:ALL')
    .config('spark.sql.catalog.polaris.credential', credentials['polaris_client_id'] + ':' + credentials['polaris_client_secret'])
    .config('spark.sql.catalog.polaris.io-impl', 'org.apache.iceberg.aws.s3.S3FileIO')
    .config('spark.sql.catalog.polaris.s3.endpoint', config['s3_endpoint'])
    .config('spark.sql.catalog.polaris.s3.path-style-access', 'true')
    .config('spark.sql.catalog.polaris.s3.access-key-id', credentials['s3_access_key'])
    .config('spark.sql.catalog.polaris.s3.secret-access-key', credentials['s3_secret_key'])
    .config('spark.sql.catalog.polaris.client.region', 'us-east-1'))
spark = builder.getOrCreate()
spark.sparkContext.setLogLevel('WARN')
assert spark.sparkContext.master == config['spark_master'], 'Restart this kernel: an old Spark context was reused'
application_id = spark.sparkContext.applicationId
print('Application:', application_id, 'Driver UI:', spark.sparkContext.uiWebUrl)


In [ ]:
orders = spark.createDataFrame([(1,'north',120),(2,'north',80),(3,'south',50),(4,'south',150)], 'id long, region string, amount long')
orders.repartition(4).write.mode('errorifexists').parquet(root + '/orders')
loaded = spark.read.parquet(root + '/orders')
assert loaded.orderBy('id').collect() == orders.orderBy('id').collect()
summary = loaded.groupBy('region').sum('amount').orderBy('region')
summary.show()
assert [(r[0],r[1]) for r in summary.collect()] == [('north',200),('south',200)]
print('PASS: four exact rows, two regional totals of 200')


In [ ]:
import logging
# This is the Python application's driver log, separate from JVM/kernel stderr.
driver_log = Path.cwd() / (run_id + '-driver.log')
logger = logging.getLogger(run_id)
logger.setLevel(logging.INFO)
logger.propagate = False
file_handler = logging.FileHandler(driver_log)
file_handler.setFormatter(logging.Formatter('%(levelname)s %(message)s'))
logger.addHandler(file_handler)
logger.info('application=%s started', application_id)
try:
    1 / 0
except ZeroDivisionError:
    logger.exception('Controlled driver-side error; Spark session remains usable')
file_handler.flush()
assert 'ZeroDivisionError' in driver_log.read_text()
assert spark.range(10).count() == 10
logger.info('application=%s healthy after handled error', application_id)
file_handler.flush()
print('PASS: driver error captured and Spark health checked')
print('Driver log:', driver_log)
print(driver_log.read_text())


Checkpoint: inspect Jobs, Stages, SQL and Executors in the live driver UI. Match the application ID in the master UI. Run the final cell only after inspection.


In [ ]:
record = {'run_id': run_id, 'application_id': application_id, 'root': root}
Path(run_id + '.json').write_text(json.dumps(record, indent=2))
spark.stop()
print('PASS: session stopped; inspect the completed application in Spark History Server')
